In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Analisis_Scraping") \
    .config("spark.mongodb.input.uri", "mongodb://database:27017/proyecto_bigdata.datos_scraping") \
    .config("spark.mongodb.output.uri", "mongodb://database:27017/proyecto_bigdata.datos_scraping") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:3.0.1") \
    .getOrCreate()

df = spark.read.format("mongo").load()

df.printSchema()
df.show(5, truncate=False)


root
 |-- _id: struct (nullable = true)
 |    |-- oid: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- identificador: string (nullable = true)
 |-- valor: double (nullable = true)

+--------------------------+-------------------+---------------+-------------------------------------+-----+
|_id                       |fecha_captura      |grupo          |identificador                        |valor|
+--------------------------+-------------------+---------------+-------------------------------------+-----+
|{69cd106f6761edfb2e26384d}|2026-04-01 12:32:39|Mercado Laboral|A Light in the Attic                 |51.77|
|{69cd106f6761edfb2e26384e}|2026-04-01 12:32:39|Mercado Laboral|Tipping the Velvet                   |53.74|
|{69cd106f6761edfb2e26384f}|2026-04-01 12:32:39|Mercado Laboral|Soumission                           |50.1 |
|{69cd106f6761edfb2e263850}|2026-04-01 12:32:39|Mercado Laboral|Sharp Objects                        

In [2]:
from pyspark.sql.functions import col

# Filtrar registros válidos
df_filtrado = df.filter(
    (col("identificador").isNotNull()) & 
    (col("valor") > 0)
)

print("PASO 3: Limpieza completada.")
print("Registros originales:", df.count())
print("Registros válidos:", df_filtrado.count())


PASO 3: Limpieza completada.
Registros originales: 60
Registros válidos: 60


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, to_timestamp

# 1. Crear sesión de Spark
spark = SparkSession.builder \
    .appName("Limpieza_Datos_Scraping") \
    .config("spark.mongodb.input.uri", "mongodb://database:27017/proyecto_bigdata.datos_scraping") \
    .config("spark.mongodb.output.uri", "mongodb://database:27017/proyecto_bigdata.datos_scraping") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:3.0.1") \
    .getOrCreate()

# 2. Cargar datos desde MongoDB
df = spark.read.format("mongo").load()

print("Registros originales:", df.count())

# 3. Proceso de limpieza completo
df_limpio = df \
    .withColumn("_id", col("_id.oid")) \
    .withColumn("identificador", trim(col("identificador"))) \
    .withColumn("fecha_captura", to_timestamp("fecha_captura", "yyyy-MM-dd HH:mm:ss")) \
    .dropna(subset=["identificador", "valor"]) \
    .filter(col("identificador") != "") \
    .filter(col("valor") > 0) \
    .dropDuplicates(["identificador"])

# 4. Resultados finales
print("Registros limpios:", df_limpio.count())

df_limpio.printSchema()
df_limpio.show(10, truncate=False)


Registros originales: 60
Registros limpios: 60
root
 |-- _id: string (nullable = true)
 |-- fecha_captura: timestamp (nullable = true)
 |-- grupo: string (nullable = true)
 |-- identificador: string (nullable = true)
 |-- valor: double (nullable = true)

+------------------------+-------------------+---------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|_id                     |fecha_captura      |grupo          |identificador                                                                                                                                                                  |valor|
+------------------------+-------------------+---------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|69cd106f6

In [4]:
df_limpio.select("identificador", "valor").show(truncate=False)


+----------------------------------------------------------------------------------+-----+
|identificador                                                                     |valor|
+----------------------------------------------------------------------------------+-----+
|Starving Hearts (Triangular Trade Trilogy, #1)                                    |13.99|
|Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)                           |52.29|
|Tsubasa: WoRLD CHRoNiCLE 2 (Tsubasa WoRLD CHRoNiCLE #2)                           |16.28|
|It's Only the Himalayas                                                           |45.17|
|The Elephant Tree                                                                 |23.82|
|Birdsong: A Story in Pictures                                                     |54.64|
|The Natural History of Us (The Fine Art of Pretending #2)                         |45.22|
|Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991|57.25|

In [5]:
df_limpio.filter(df_limpio["valor"] > 50).show(truncate=False)


+------------------------+-------------------+---------------+-------------------------------------------------------------------------------------------------------------------------------+-----+
|_id                     |fecha_captura      |grupo          |identificador                                                                                                                  |valor|
+------------------------+-------------------+---------------+-------------------------------------------------------------------------------------------------------------------------------+-----+
|69cd106f6761edfb2e26384d|2026-04-01 12:32:39|Mercado Laboral|A Light in the Attic                                                                                                           |51.77|
|69cd106f6761edfb2e263868|2026-04-01 12:32:43|Mercado Laboral|Aladdin and His Wonderful Lamp                                                                                                 |53.13|
|69cd106f6761ed

In [6]:
df_limpio.groupBy("grupo").count().show()


+---------------+-----+
|          grupo|count|
+---------------+-----+
|Mercado Laboral|   60|
+---------------+-----+



In [7]:
from pyspark.sql import functions as F

reporte_grupo = df_limpio.groupBy("grupo").agg(
    F.count("identificador").alias("cantidad_productos"),
    F.avg("valor").alias("precio_promedio"),
    F.min("valor").alias("precio_minimo"),
    F.max("valor").alias("precio_maximo")
).orderBy(F.desc("precio_promedio"))

print("ANÁLISIS DE MERCADO POR GRUPO:")
reporte_grupo.show()


ANÁLISIS DE MERCADO POR GRUPO:
+---------------+------------------+-----------------+-------------+-------------+
|          grupo|cantidad_productos|  precio_promedio|precio_minimo|precio_maximo|
+---------------+------------------+-----------------+-------------+-------------+
|Mercado Laboral|                60|35.00266666666667|        12.84|        57.31|
+---------------+------------------+-----------------+-------------+-------------+



In [8]:
from pyspark.sql import functions as F

# 1. Ordenar por precio (valor) de mayor a menor
ganador = df_limpio.orderBy(F.desc("valor")).limit(1)

# 2. Mostrar la información solicitada
ganador.select(
    "identificador",
    "valor",
    "grupo",
    "fecha_captura"
).show(truncate=False)


+------------------------------+-----+---------------+-------------------+
|identificador                 |valor|grupo          |fecha_captura      |
+------------------------------+-----+---------------+-------------------+
|Slow States of Collapse: Poems|57.31|Mercado Laboral|2026-04-01 12:32:45|
+------------------------------+-----+---------------+-------------------+



In [ ]:
from pyspark.sql import functions as F

NOMBRE_GRUPO = "Mercado Laboral"

ticket_salida = df_limpio.filter(F.col("grupo") == NOMBRE_GRUPO) \
    .agg(
        F.count("identificador").alias("total_productos"),
        F.round(F.avg("valor"), 2).alias("precio_medio"),
        F.max("fecha_captura").alias("ultima_sincronizacion")
    )

print(f"--- TICKET DE SALIDA: {NOMBRE_GRUPO} ---")
ticket_salida.show(truncate=False)
